# 04 Evaluate

1. Reproduces the wallet classifiers of Elmougy and Liu on the recovered lifetime features and on the released features.
2. Repeats the reproduction with the scaler fitted over all wallet addresses, including those of unknown class.
3. Measures the sensitivity of the reproduction to the divergent addresses of `03`.
4. Separates the two leakage channels, the split and the feature computation.

LR is fitted once with the default L-BFGS solver and L2 penalty of scikit-learn. RF has 50 trees and is fitted with
seeds 66 to 75; its precision, recall, and F1 are reported as mean and standard deviation over the seeds.
All metrics are for the illicit class at the default threshold of 0.5.

The positional split depends on the row order of the wallet file, which is kept as released. All other inputs are sorted by address.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

ROOT      = Path.cwd().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS   = ROOT / 'outputs'
WALLETS   = RAW / 'elliptic_pp' / 'wallets_features_classes_combined.csv'
FEATURES_LIFETIME  = PROCESSED / 'features_lifetime.parquet'
FEATURES_RESISTANT = PROCESSED / 'features_leakage_resistant.parquet'
pd.set_option('display.width', 200)

STATS = ['total', 'min', 'max', 'mean', 'median']
FAMILIES = {
    'Volumes':      [f'btc_{q}_{s}' for q in ['transacted', 'sent', 'received'] for s in STATS],
    'Fees':         [f'fees_{s}' for s in STATS] + [f'fees_as_share_{s}' for s in STATS],
    'Block gaps':   [f'blocks_btwn_{q}_{s}' for q in ['txs', 'input_txs', 'output_txs'] for s in STATS],
    'Counterparty': [f'transacted_w_address_{s}' for s in STATS] + ['num_addr_transacted_multiple'],
    'Counts':       ['total_txs', 'num_txs_as_sender', 'num_txs_as receiver'],
    'Lifetime':     ['first_block_appeared_in', 'last_block_appeared_in', 'lifetime_in_blocks',
                     'first_sent_block', 'first_received_block', 'num_timesteps_appeared_in'],
}
FEATURES = [f for fs in FAMILIES.values() for f in fs]
assert len(FEATURES) == 55

con = duckdb.connect()
released = con.execute(f"""
    SELECT * FROM read_csv('{WALLETS}', header=true) WHERE "class" IN (1, 2)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY address ORDER BY "Time step") = 1
    ORDER BY address""").fetchdf()
lifetime  = pd.read_parquet(FEATURES_LIFETIME).sort_values('address').reset_index(drop=True)
resistant = pd.read_parquet(FEATURES_RESISTANT).sort_values('address').reset_index(drop=True)
print(f'released {len(released):,}  lifetime {len(lifetime):,}  leakage-resistant {len(resistant):,}')

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support

SEEDS = range(66, 76)
REPORTED = pd.DataFrame({'P': [0.477, 0.911], 'R': [0.046, 0.789], 'F1': [0.083, 0.845]}, index=['LR', 'RF'])


def prf(y_true, y_pred):
    return precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)[:3]


def fit_score(X_train, y_train, X_test, y_test):
    rows = [('LR', *prf(y_test, LogisticRegression(max_iter=1000).fit(X_train, y_train).predict(X_test)))]
    for seed in SEEDS:
        rf = RandomForestClassifier(n_estimators=50, random_state=seed, n_jobs=-1).fit(X_train, y_train)
        rows.append(('RF', *prf(y_test, rf.predict(X_test))))
    r = pd.DataFrame(rows, columns=['model', 'P', 'R', 'F1']).groupby('model')
    return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


def positional(X, y):
    Xs = MinMaxScaler().fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(Xs, y, test_size=0.30, shuffle=False)
    return fit_score(X_train, y_train, X_test, y_test)


def temporal(df):
    tr, te = df[df.split == 'train'], df[df.split == 'test']
    scaler = MinMaxScaler().fit(tr[FEATURES].to_numpy(float))
    return fit_score(scaler.transform(tr[FEATURES].to_numpy(float)), (tr.label == 1).to_numpy(int),
                     scaler.transform(te[FEATURES].to_numpy(float)), (te.label == 1).to_numpy(int))

released 265,354  lifetime 265,352  leakage-resistant 265,352


## Rows of the original pipeline

The wallet file has one row per address and time step. The pipeline of Elmougy and Liu drops the time step, removes
duplicate rows, and keeps the labelled addresses in file order. The two addresses without a ledger record are dropped,
so that the recovered and the released features are scored on the same rows.

In [2]:
wallets = pd.read_csv(WALLETS)
dedup = wallets.drop(columns=['Time step']).drop_duplicates()
rows = dedup[dedup['class'] != 3]
rows = rows[rows.address.isin(set(lifetime.address))].reset_index(drop=True)
y = (rows['class'] == 1).to_numpy(int)
X_released  = rows[FEATURES].to_numpy(float)
X_recovered = lifetime.set_index('address').loc[rows.address, FEATURES].to_numpy(float)
print(f'rows {len(rows):,}  illicit {y.sum():,}  addresses {rows.address.nunique():,}')

rows 265,352  illicit 14,266  addresses 265,352


## 1. Reproduction

Positional 70/30 split, MinMax scaling over the labelled addresses (Table `tab:reproduction`).

In [3]:
reproduction = pd.concat({'recovered': positional(X_recovered, y),
                          'released':  positional(X_released, y),
                          'Elmougy and Liu': REPORTED}, axis=1)
print(reproduction.to_string())

/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


   recovered                                    released                                    Elmougy and Liu              
           P      R     F1   P sd   R sd  F1 sd        P      R     F1   P sd   R sd  F1 sd               P      R     F1
LR     0.492  0.062  0.110    NaN    NaN    NaN    0.499  0.062  0.110    NaN    NaN    NaN           0.477  0.046  0.083
RF     0.911  0.786  0.844  0.004  0.003  0.002    0.916  0.787  0.846  0.004  0.003  0.002           0.911  0.789  0.845


/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


## 2. Scaling over all wallet addresses

As in the original pipeline, the scaler is fitted on all deduplicated wallet rows, including those of unknown class,
before the unknown rows are removed.

In [4]:
Xs_all = MinMaxScaler().fit_transform(dedup[FEATURES].to_numpy(float))
keep = (dedup['class'] != 3).to_numpy()
Xs, y_all = Xs_all[keep], (dedup.loc[keep, 'class'] == 1).to_numpy(int)
X_train, X_test, y_train, y_test = train_test_split(Xs, y_all, test_size=0.30, shuffle=False)
print(f'rows scaled {len(dedup):,}, labelled rows {keep.sum():,}')
print(fit_score(X_train, y_train, X_test, y_test).to_string())

rows scaled 822,942, labelled rows 265,354
           P      R     F1   P sd   R sd  F1 sd
model                                          
LR     0.491  0.057  0.102    NaN    NaN    NaN
RF     0.910  0.785  0.843  0.004  0.002  0.002


/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


## 3. Sensitivity to the divergent addresses

Recovered features, positional split, without the addresses found divergent in `03` (Table `tab:divergent-sensitivity`).

In [5]:
results = {'all addresses': positional(X_recovered, y)}
for name in ['1e-4', '1e-8']:
    drop = set(pd.read_parquet(PROCESSED / f'divergent_{name}.parquet').address)
    keep = ~rows.address.isin(drop).to_numpy()
    results[f'{(~keep).sum()} dropped (floor {name})'] = positional(X_recovered[keep], y[keep])
print(pd.concat(results, axis=1).to_string())

/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')
/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


      all addresses                                    86 dropped (floor 1e-4)                                    137 dropped (floor 1e-8)                                   
                  P      R     F1   P sd   R sd  F1 sd                       P      R     F1   P sd   R sd  F1 sd                        P      R     F1   P sd   R sd  F1 sd
model                                                                                                                                                                        
LR            0.492  0.062  0.110    NaN    NaN    NaN                   0.481  0.065  0.114    NaN    NaN    NaN                    0.483  0.064  0.114    NaN    NaN    NaN
RF            0.911  0.786  0.844  0.004  0.003  0.002                   0.915  0.786  0.846  0.004  0.002  0.002                    0.912  0.785  0.844  0.005  0.003  0.004


/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


## 4. Effect of leakage

Each address enters one side of the temporal split, by the time step of its first appearance (`01`).
The scaler is fitted on training addresses only (Table `tab:leakage`).

In [6]:
split = lifetime[['address', 'label', 'split']]
released_split = released[['address'] + FEATURES].merge(split, on='address').sort_values('address')
leakage = pd.concat({'positional, released':          reproduction['released'],
                     'temporal, released':            temporal(released_split),
                     'temporal, leakage-resistant':   temporal(resistant)}, axis=1)
print(leakage.to_string())

/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


   positional, released                                    temporal, released                                    temporal, leakage-resistant                                  
                      P      R     F1   P sd   R sd  F1 sd                  P      R     F1   P sd   R sd  F1 sd                           P      R     F1   P sd  R sd  F1 sd
LR                0.499  0.062  0.110    NaN    NaN    NaN              0.199  0.520  0.288    NaN    NaN    NaN                       0.198  0.506  0.285    NaN   NaN    NaN
RF                0.916  0.787  0.846  0.004  0.003  0.002              0.381  0.196  0.257  0.048  0.033  0.035                       0.407  0.228  0.291  0.074  0.05  0.058


/scratch-local/rliang.28508984/ipykernel_373052/1594359103.py:57: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  return r.mean().round(3).join(r.std().round(3), rsuffix=' sd')


In [7]:
share = split.groupby('split').label.agg(addresses='size', illicit=lambda s: (s == 1).sum())
share['illicit share'] = (share.illicit / share.addresses).round(4)
print(share.loc[['train', 'test']].to_string())

       addresses  illicit  illicit share
split                                   
train     172903     9377         0.0542
test       92449     4889         0.0529
